In [3]:
!pip install lightgbm optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [4]:
import pandas as pd, numpy as np
df = pd.read_parquet("candidates.parquet")
FEATS = [c for c in df.columns if c.startswith("f_")]
tr, te = df[df.split=="train"], df[df.split=="student_test"]
X, y, g = tr[FEATS].values, tr.label.values, tr.trip_id.values
print(y.mean())   # probably ~0.05 → only 1 in ~20 rows is "yes"

0.11144430543577219


In [5]:
from sklearn.metrics import f1_score, log_loss, precision_score, recall_score, confusion_matrix

def top1(frame, proba):
    f = frame.assign(p=proba)
    best = f.loc[f.groupby(["trip_id","obs_idx"]).p.idxmax()]
    return best.label.mean()      # fraction of GPS points where the top pick is correct

def score(frame, proba, thr=0.5):
    yt, yp = frame.label.values, (proba >= thr).astype(int)
    return dict(macro_f1=f1_score(yt, yp, average="macro"),
                precision=precision_score(yt, yp), recall=recall_score(yt, yp),
                log_loss=log_loss(yt, proba), top1=top1(frame, proba))

In [6]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import time

gkf = GroupKFold(n_splits=5)
def run_cv(make_model):
    fold_scores = []
    for tri, vai in gkf.split(X, y, g):
        m = make_model()
        m.fit(X[tri], y[tri])
        p = m.predict_proba(X[vai])[:,1]
        fold_scores.append(score(tr.iloc[vai], p))
    return pd.DataFrame(fold_scores)   # mean = performance, std = stability

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
S = 42
MODELS = {
 "LogReg":   lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
 "KNN":      lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25, weights="distance")),
 "RF":       lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=S),
 "LightGBM": lambda: LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, class_weight="balanced", random_state=S, verbose=-1),
 "MLP":      lambda: make_pipeline(StandardScaler(), MLPClassifier((64,32), early_stopping=True, random_state=S)),
 "GNB":      lambda: make_pipeline(StandardScaler(), GaussianNB()),
}

In [9]:
import os, glob, zipfile
from functools import lru_cache
import numpy as np
import pandas as pd

ZIP_PATH = "/content/Student folder.zip"
EXTRACT_TO = "/content/data"

# 1. Unzip the dataset (only the first time)
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"{ZIP_PATH} not found - upload 'Student folder.zip' using the 📁 sidebar first")
if not os.path.exists(EXTRACT_TO):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_TO)

# 2. Find the folder that contains train/ and student_test/ (handles double folders)
hits = glob.glob(f"{EXTRACT_TO}/**/train", recursive=True)
if not hits:
    raise FileNotFoundError("No 'train' folder found inside the zip")
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT)
print("train trips:", len(os.listdir(f"{DATA_ROOT}/train")),
      "| student_test trips:", len(os.listdir(f"{DATA_ROOT}/student_test")))

# 3. Person 3's continuity function, with the corrected path
@lru_cache(maxsize=None)
def _arc_ends(trip_id, split):
    trip_id = str(trip_id).zfill(8)
    arcs = pd.read_csv(f"{DATA_ROOT}/{split}/{trip_id}/{trip_id}.arcs", sep=r"\s+",
                       header=None, names=["from_node", "to_node"])
    return arcs["from_node"].to_numpy(), arcs["to_node"].to_numpy()

def continuity(trip_id, split, pred_arcs):
    """Fraction of consecutive GPS points whose predicted arcs are the same
    or connected (end node of arc i == start node of arc i+1)."""
    a = np.asarray(pred_arcs)
    if len(a) < 2:
        return float("nan")
    frm, to = _arc_ends(trip_id, split)
    prev, nxt = a[:-1], a[1:]
    n = len(frm)
    valid = (prev >= 0) & (prev < n) & (nxt >= 0) & (nxt < n)
    ok = np.zeros(len(prev), dtype=bool)
    p, q = prev[valid], nxt[valid]
    ok[valid] = (p == q) | (to[p] == frm[q])
    return float(ok.mean())

# 4. Helper for your models: top pick per GPS point -> average continuity over trips
def route_continuity(frame, proba, split):
    best = frame.assign(p=proba).loc[lambda f: f.groupby(["trip_id", "obs_idx"]).p.idxmax()]
    best = best.sort_values(["trip_id", "obs_idx"])
    per_trip = {t: continuity(t, split, g.cand_arc.values) for t, g in best.groupby("trip_id")}
    return float(np.nanmean(list(per_trip.values()))), per_trip

# 5. Sanity check: the true route should give ~0.98
pts = pd.read_parquet("/content/points.parquet").sort_values(["trip_id", "obs_idx"])
st = pts[pts.split == "student_test"]
print("reference-route continuity (student_test):",
      round(np.mean([continuity(t, "student_test", g.true_arc.values) for t, g in st.groupby("trip_id")]), 4))

DATA_ROOT = /content/data/Student folder
train trips: 80 | student_test trips: 10
reference-route continuity (student_test): 0.9808


In [10]:
!pip -q install lightgbm

In [11]:
import time, json, os
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             log_loss, confusion_matrix)

SEED = 42
os.makedirs("/content/results", exist_ok=True)

df = pd.read_parquet("/content/candidates.parquet")
print("all rows:", df.shape)

# keep only GPS points whose reference label is trustworthy (<= 50 m from its reference arc)
df = df[df.label_ok].reset_index(drop=True)
print("after label_ok filter:", df.shape)

FEATS = [c for c in df.columns if c.startswith("f_")]
print("features:", FEATS)

tr = df[df.split == "train"].reset_index(drop=True)
te = df[df.split == "student_test"].reset_index(drop=True)
X_tr, y_tr, g_tr = tr[FEATS].to_numpy(), tr.label.to_numpy(), tr.trip_id.to_numpy()
X_te, y_te = te[FEATS].to_numpy(), te.label.to_numpy()

print(f"train: {len(tr):,} rows, {tr.trip_id.nunique()} trips, positive rate {y_tr.mean():.3f}")
print(f"test : {len(te):,} rows, {te.trip_id.nunique()} trips, positive rate {y_te.mean():.3f}")

all rows: (1851079, 17)
after label_ok filter: (1772979, 17)
features: ['f_dist', 'f_proj_frac', 'f_dist_rank', 'f_heading_diff', 'f_heading_valid', 'f_arc_len', 'f_n_cands', 'f_dist_rel', 'f_speed', 'f_dt_prev', 'f_no_cand_in_radius']
train: 1,589,322 rows, 80 trips, positive rate 0.116
test : 183,657 rows, 10 trips, positive rate 0.104


In [12]:
def top1_accuracy(frame, proba):
    """For each GPS point, take the candidate with the highest probability
    and check whether it is the true road."""
    f = frame[["trip_id", "obs_idx", "label"]].assign(p=proba)
    best = f.loc[f.groupby(["trip_id", "obs_idx"]).p.idxmax()]
    return best.label.mean()

def all_scores(frame, proba, thr=0.5):
    y = frame.label.to_numpy()
    pred = (proba >= thr).astype(int)
    return dict(macro_f1=f1_score(y, pred, average="macro"),
                precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred),
                log_loss=log_loss(y, np.clip(proba, 1e-7, 1 - 1e-7)),
                top1=top1_accuracy(frame, proba))

gkf = GroupKFold(n_splits=5)   # every trip stays entirely in train OR validation

def sample_rows(n, frac):
    if frac >= 1: return np.arange(n)
    rng = np.random.default_rng(SEED)
    return np.sort(rng.choice(n, int(n * frac), replace=False))

def run_cv(make_model, train_frac=1.0):
    folds = []
    for k, (tri, vai) in enumerate(gkf.split(X_tr, y_tr, g_tr), 1):
        tri = tri[sample_rows(len(tri), train_frac)]
        m = make_model()
        m.fit(X_tr[tri], y_tr[tri])
        p = m.predict_proba(X_tr[vai])[:, 1]
        s = all_scores(tr.iloc[vai], p); s["fold"] = k
        folds.append(s)
        print(f"   fold {k}: macro_f1={s['macro_f1']:.4f} top1={s['top1']:.4f}")
    return pd.DataFrame(folds)

In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

# name: (family, model factory, fraction of training rows used, config text for the report)
MODELS = {
 "LogReg":   ("Linear", lambda: make_pipeline(StandardScaler(),
                 LogisticRegression(class_weight="balanced", max_iter=1000)), 1.0,
              "C=1.0, balanced weights, scaled"),
 "GaussNB":  ("Probabilistic", lambda: make_pipeline(StandardScaler(), GaussianNB()), 1.0,
              "default var_smoothing, scaled"),
 "KNN":      ("Instance-based", lambda: make_pipeline(StandardScaler(),
                 KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)), 0.05,
              "k=25, distance-weighted, scaled, 5% row sample"),
 "RandomForest": ("Ensemble (bagging)", lambda: RandomForestClassifier(
                 n_estimators=100, min_samples_leaf=10, max_features="sqrt", max_samples=0.2,
                 class_weight="balanced", n_jobs=-1, random_state=SEED), 1.0,
              "100 trees, min_leaf=10, 20% bootstrap per tree, balanced"),
 "LightGBM": ("Boosting", lambda: LGBMClassifier(
                 n_estimators=400, learning_rate=0.05, num_leaves=31,
                 class_weight="balanced", random_state=SEED, verbose=-1), 1.0,
              "400 trees, lr=0.05, 31 leaves, balanced"),
 "MLP":      ("Neural network", lambda: make_pipeline(StandardScaler(),
                 MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                               max_iter=50, random_state=SEED)), 0.3,
              "64-32 ReLU, adam, early stopping, 30% row sample"),
}

In [19]:
import joblib, os
RES = "/content/results"
os.makedirs(f"{RES}/models", exist_ok=True)
comp_path = f"{RES}/model_comparison.csv"
results = pd.read_csv(comp_path).to_dict("records") if os.path.exists(comp_path) else []
done = {r["model"] for r in results}
print("already done:", done or "none")

for name, (family, make, frac, cfg) in MODELS.items():
    if name in done:
        continue
    print(f"\n=== {name} ===")
    t0 = time.time()
    cv = run_cv(make, frac)
    cv.to_csv(f"{RES}/cv_folds_{name}.csv", index=False)

    idx = sample_rows(len(X_tr), frac)
    m = make()
    t1 = time.time(); m.fit(X_tr[idx], y_tr[idx]); train_time = time.time() - t1
    t1 = time.time(); p = m.predict_proba(X_te)[:, 1]; infer_time = time.time() - t1
    s = all_scores(te, p)
    cont, _ = route_continuity(te, p, "student_test")
    cm = confusion_matrix(y_te, (p >= 0.5).astype(int))

    row = dict(model=name, family=family, config=cfg, train_frac=frac,
               cv_macro_f1=cv.macro_f1.mean(), cv_macro_f1_std=cv.macro_f1.std(),
               cv_top1=cv.top1.mean(), cv_top1_std=cv.top1.std(), cv_log_loss=cv.log_loss.mean(),
               test_macro_f1=s["macro_f1"], test_top1=s["top1"], test_log_loss=s["log_loss"],
               test_precision=s["precision"], test_recall=s["recall"], test_continuity=cont,
               train_time_s=train_time, infer_ms_per_1k_rows=1000 * infer_time / len(X_te) * 1000,
               total_time_s=time.time() - t0, tn=cm[0,0], fp=cm[0,1], fn=cm[1,0], tp=cm[1,1])
    results.append(row)
    pd.DataFrame(results).to_csv(comp_path, index=False)      # table so far
    np.save(f"{RES}/test_proba_{name}.npy", p)                 # probabilities for Person 3
    joblib.dump(m, f"{RES}/models/{name}.joblib")              # fitted model
    print(f"saved {name}: test_top1={s['top1']:.4f}  macro_f1={s['macro_f1']:.4f}  ({time.time()-t0:.0f}s)")

comp = pd.DataFrame(results)
comp[["model", "cv_macro_f1", "cv_top1", "test_macro_f1", "test_top1",
      "test_log_loss", "test_continuity", "train_time_s"]].round(4)

already done: {'GaussNB', 'KNN', 'LogReg'}

=== RandomForest ===
   fold 1: macro_f1=0.8888 top1=0.8383
   fold 2: macro_f1=0.9008 top1=0.8432
   fold 3: macro_f1=0.8792 top1=0.8006
   fold 4: macro_f1=0.9163 top1=0.8833
   fold 5: macro_f1=0.8917 top1=0.8453
saved RandomForest: test_top1=0.8546  macro_f1=0.9021  (617s)

=== LightGBM ===


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   fold 1: macro_f1=0.8669 top1=0.8392


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


   fold 2: macro_f1=0.8780 top1=0.8439


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


... (6 outputs left out)

saved LightGBM: test_top1=0.8584  macro_f1=0.8797  (297s)

=== MLP ===
   fold 1: macro_f1=0.8980 top1=0.8361
   fold 2: macro_f1=0.9084 top1=0.8407
   fold 3: macro_f1=0.8765 top1=0.7982


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


   fold 4: macro_f1=0.9284 top1=0.8809
   fold 5: macro_f1=0.9014 top1=0.8396
saved MLP: test_top1=0.8525  macro_f1=0.9125  (608s)


,model,cv_macro_f1,cv_top1,test_macro_f1,test_top1,test_log_loss,test_continuity,train_time_s
0,LogReg,0.8088,0.8349,0.8123,0.8472,0.2065,0.9119,6.3935
1,GaussNB,0.7803,0.8195,0.7849,0.8193,0.4388,0.9009,0.9246
2,KNN,0.8924,0.8281,0.9047,0.8452,0.1255,0.9109,0.1575
3,RandomForest,0.8954,0.8421,0.9021,0.8546,0.1129,0.9199,122.0926
4,LightGBM,0.8747,0.8436,0.8797,0.8584,0.1370,0.9196,52.2572
5,MLP,0.9025,0.8391,0.9125,0.8525,0.0851,0.9187,128.8179


In [23]:
import joblib
RES = "/content/results"
os.makedirs(f"{RES}/models", exist_ok=True)
test_proba = globals().get("test_proba", {})
models_fitted = globals().get("models_fitted", {})

probs = te[["trip_id", "obs_idx", "cand_arc", "label"]].copy()
for name, (family, make, frac, cfg) in MODELS.items():
    if name in test_proba:
        src = "memory"
    else:
        path = f"{RES}/models/{name}.joblib"
        if name in models_fitted:
            m, src = models_fitted[name], "memory model"
        elif os.path.exists(path):
            m, src = joblib.load(path), "saved file"
        else:
            idx = sample_rows(len(X_tr), frac)
            m = make(); m.fit(X_tr[idx], y_tr[idx]); src = "refit"
        models_fitted[name] = m
        test_proba[name] = m.predict_proba(X_te)[:, 1]
    probs[f"p_{name}"] = test_proba[name]
    np.save(f"{RES}/test_proba_{name}.npy", test_proba[name])
    print(f"{name}: {src}")

probs.to_parquet(f"{RES}/test_probabilities.parquet", index=False)
joblib.dump(models_fitted["LightGBM"], f"{RES}/models/LightGBM.joblib")
json.dump({"model": "LightGBM", "features": FEATS, "note": "interim - untuned"},
          open(f"{RES}/interim_model_config.json", "w"), indent=2)
print("done:", probs.shape)

LogReg: refit
GaussNB: refit
KNN: refit
RandomForest: saved file


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM: saved file
MLP: saved file
done: (183657, 10)


In [24]:
from google.colab import files
files.download("/content/results/test_probabilities.parquet")
files.download("/content/results/models/LightGBM.joblib")
files.download("/content/results/interim_model_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>